In [ ]:
%reset -f

In [ ]:
import pypsa
import pandas as pd
import os
import matplotlib.pyplot as plt

In [ ]:
data_path = os.path.join(os.getcwd(), 'data')

In [ ]:
#Importing network data from excel files; consider moving to csv's

buses_df = pd.read_excel(os.path.join(data_path, 'buses.xlsx'), index_col = 0)
buses_df = buses_df.drop(['MV_bus_2'] ) #dropping the second MV bus, will add back when dealing with switching behaviour

gens_df = pd.read_excel(os.path.join(data_path, 'gens.xlsx'), index_col = 0)
gens_pu_df = pd.read_excel(os.path.join(data_path, 'gen_profiles_pu.xlsx'), index_col = 0)

links_df = pd.read_excel(os.path.join(data_path, 'links.xlsx'), index_col = 0)

loads_df = pd.read_excel(os.path.join(data_path, 'loads.xlsx'), index_col = 0)
loads_pu_df = pd.read_excel(os.path.join(data_path, 'load_profiles_pu.xlsx'), index_col = 0)


In [ ]:
# Adding thermal buses at same locations as electrical buses, to represent thermal energy flows.
thermal_buses_df = buses_df.copy(deep = True)
thermal_buses_df.index = thermal_buses_df.index + '_thermal'
thermal_buses_df['carrier'] = 'thermal'
thermal_buses_df = thermal_buses_df.drop(columns = ['v_nom'])
thermal_buses_df = thermal_buses_df.drop(index = ['Extern_grid_thermal'])
thermal_buses_df

Pypsa resolves carriers by carrier of buses, carriers on gens and loads, are just for grouping and statistics, and don't mean much.  
What matters is what is connected to which bus!!

In [ ]:
# adding gas buses at same locations as electrical buses, to represent gas energy flows.
gas_buses_df = buses_df.copy(deep = True)
 
gas_buses_df.index = gas_buses_df.index + '_gas'
gas_buses_df.index = gas_buses_df.index.str.replace('Extern_grid_gas', 'Gas_substation')
gas_buses_df.loc['Gas_substation', 'x'] = gas_buses_df.loc['Gas_substation', 'x'] - 0.01 #jsut offsetting it a bit, might be usefule later while visualizing
gas_buses_df['carrier'] = 'gas'
gas_buses_df = gas_buses_df.drop(columns = ['v_nom'])
gas_buses_df

In [ ]:
def clean_index(snaps, df):
    """
    Function to clean the index of a dataframe to match the snapshots of the network.
    This is useful when the index of the dataframe does not match the snapshots of the network.
    Will sort out small float point erros, and also point if big errors
    """
    df.index = pd.to_datetime(df.index)
    missing_idxs = set(snaps) - set(df.index)
    if len(missing_idxs) > 0:
        print(f"{set(snaps) - set(df.index)} missing from dataframe index.")
    if len(df.index) != len(snaps):
        raise ValueError(f"Length of dataframe index ({len(df.index)}) does not match length of snapshots ({len(snaps)}).")
    else:
        print(f"Length of both match, overwriting dataframe index with snapshots.")
        df.index = snaps
    return df

In [ ]:
n = pypsa.Network()

# n.set_snapshots(pd.date_range('2026-01-01 10:00:00', '2026-01-01 12:00:00', freq = 'h'))
n.set_snapshots(pd.date_range('2026-01-01 00:00:00', '2026-12-31 23:00:00', freq = 'h'))
## verify index of load/gen profile input datasets
loads_pu_df = clean_index(n.snapshots, loads_pu_df)
gens_pu_df = clean_index(n.snapshots, gens_pu_df)

# adding buses
n.add('Bus', buses_df.index, v_nom = buses_df['v_nom'], x = buses_df['x'], y = buses_df['y'], )
n.add('Bus', thermal_buses_df.index, x = thermal_buses_df['x'], y = thermal_buses_df['y'], carrier = thermal_buses_df['carrier'])
n.add('Bus', gas_buses_df.index, x = gas_buses_df['x'], y = gas_buses_df['y'], carrier = gas_buses_df['carrier'])

#adding gens

n.add('Generator', gens_df.index, bus = gens_df['bus'], p_nom = gens_df['p_nom'], marginal_cost = gens_df['marginal_cost'], control = gens_df['control'], carrier = gens_df['carrier'])
n.generators_t.p_max_pu = gens_pu_df

#adding loads
loads_pset = loads_pu_df.multiply(loads_df['p_max'], axis = 1)
n.add('Load', loads_df.index, bus = loads_df['bus'], p_set = loads_pset)

#adding links
# links best represent abstracted DC flows, we don't want AC power dynamics, voltage angles and so on
n.add('Link', links_df.index, bus0 = links_df['bus0'], bus1 = links_df['bus1'], bus2 = links_df['bus2'], carrier = links_df['carrier'], p_nom = links_df['p_nom'], efficiency = links_df['efficiency'], efficiency2 = links_df['efficiency2'], p_min_pu = links_df['p_min_pu'])
n.links_t.efficiency['Residential_HeatPump'] = gens_pu_df['HP_COP'] #efficiency of heat pump is time dependent

# adding slack gens in each bus to represent load shedding
for bus in n.buses.index:
    n.add('Generator', f'Slack_{bus}', bus = bus, p_nom = 100, marginal_cost = 10000, control = 'Slack', carrier = 'Load_shed') #carrier helps to group later

# adding a sink to represent export of excess energy to grid
n.add('Store', 'Export_sink', bus = 'Extern_grid', e_nom = 10000, e_initial = 0, marginal_cost = 0, carrier = 'Export')


**CHP, and grid exports**  
The store gives surplus electricity somewhere to flow.  
The surplus could also be re-distributed within the grid, would depend on the costs set.  
Exact flow tracing is not essential for this study, that trouble could be neglected.  
Exports could still be considered, and appropriate export and import tariffs could be used.  


In [ ]:
def register_carriers(n):
    carriers = pd.unique(pd.concat([
        n.buses["carrier"],
        n.generators["carrier"],
        n.links["carrier"],
        n.loads["carrier"],
        n.stores["carrier"]
    ]).dropna())

    colors = {
    "AC": "tab:blue",
    "grid": "tab:cyan",
    "solar": "gold",
    "gas": "#7e1408",
    "heat": "orange",
    "GasBoiler": "tab:red",
    "Load_shed": "black",
    "CHP": "#f57c0a",
    "HP" : "#e8ff00",
    "Export" : "#ff00f6"
}
    for carrier in carriers:
        if carrier not in n.carriers.index:
            n.add("Carrier", carrier)
        n.carriers.loc[carrier, "color"] = colors.get(carrier, "tab:gray")  # Assign color or default to gray       

register_carriers(n)

In [ ]:
n.sanitize()

In [ ]:
n.optimize(), n.objective

In [ ]:
# verifying heat pump cop working well
hp_thermal = -1 * n.links_t.p1['Residential_HeatPump']

hp_env = n.links_t.p['Residential_HeatPump']
hp_thermal / hp_env


## Visualization

### Electric

In [ ]:
el_buses = n.buses[n.buses.carrier == 'AC'].index
el_gens = n.generators.loc[n.generators['bus'].isin(el_buses)]
el_gens = el_gens.loc[el_gens['carrier'] != 'Load_shed'] #removing slack gens from the list of electrical gens

el_load_shed = n.generators.loc[n.generators['carrier'] == 'Load_shed'] #:)



In [ ]:
n.stores.loc['Export_sink', 'carrier']

In [ ]:
# Determining the energy balance at each bus

## Classical generators and their contribution at each bus
gen_bus_sizes = (
    n.generators_t.p
    .sum()
    .groupby(
        [n.generators.bus, n.generators.carrier]
    )
    .sum()
)

## sector-coupling gens, and their contribution at each bus. 
link_bus_sizes = (
    n.links_t.p1
    .sum()
    .groupby(
        [n.links.bus1, n.links.carrier]
    )
    .sum()
)
link2_bus_sizes = (
    n.links_t.p2
    .sum()
    .groupby(
        [n.links.bus2, n.links.carrier]
    )
    .sum()
)

##considering only the charging into the export sink, and not the discharging from it, which is the export to grid. 
export = pd.Series(
    {
        ("Extern_grid", n.stores.loc['Export_sink', 'carrier']): -1 * n.stores_t.p["Export_sink"][n.stores_t.p["Export_sink"] < 0].sum() 
    }
)

bus_sizes = pd.concat([gen_bus_sizes, -1 * link_bus_sizes, -1 * link2_bus_sizes])
bus_sizes = bus_sizes[bus_sizes > 1e-6] #removing zeroes and very small values
bus_sizes = pd.concat([bus_sizes, export])

#normalised bus sizes, to get a sense of the relative contribution of each carrier at each bus
bus_sizes_norm = bus_sizes/bus_sizes.groupby(level = 0).sum()

In [ ]:
def plot_EnBalance(n, carrier, ax, bus_sizes, bus_scale = 3e-10, title = 'En balance'):

    #filtering buses and bus sizes for the carrier of interest
    cr_buses = n.buses[n.buses.carrier == carrier].index
    bus_sizes_cr = bus_sizes.loc[bus_sizes.index.get_level_values(0).isin(cr_buses)]
    
    #Setting link width only for carrier of interest, and the remaining zero
    link_width = pd.Series(0.0, index=n.links.index)
    link_width[n.links['carrier'] == carrier] = 1.5

    n.plot(
        ax=ax,
        bus_sizes=bus_sizes_cr * bus_scale,
        link_widths=link_width,
        geomap = False,
        title = title,
        bus_split_circle = False
    )

    # Anootation text with bus names, offset a bit
    offset = 0.00015  
    for name, row in n.buses.loc[cr_buses].iterrows():
        ax.text(
            row.x + offset,
            row.y + offset,
            name,
            fontsize=8
        )
    handles = []

    # Creating legends based on carriers
    for carrier in bus_sizes.index.get_level_values(1).unique():
        handles.append(
            plt.Line2D(
                [],
                [],
                marker="o",
                linestyle="",
                color=n.carriers.loc[carrier, "color"],
                label=carrier
            )
        )

    plt.legend(handles=handles)

In [ ]:
bus_sizes['Extern_grid']

In [ ]:
fig, ax = plt.subplots(figsize = (10,8))
plot_EnBalance(n, 'AC', ax, bus_sizes = bus_sizes, title = 'Electrical energy balance', bus_scale = 1e-10)

In [ ]:
fig, ax = plt.subplots(figsize = (10,8))
plot_EnBalance(n, 'thermal', ax, bus_sizes = bus_sizes, title = 'Thermal energy balance', bus_scale = 1e-10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 15), nrows = 3)
plot_EnBalance(n, 'AC', ax[0], bus_sizes = bus_sizes, title = 'Electrical energy balance', bus_scale = 1e-10)
plot_EnBalance(n, 'thermal', ax[1], bus_sizes = bus_sizes_norm, bus_scale = 1e-7, title = 'Thermal energy balance')
plot_EnBalance(n, 'gas', ax[2], bus_sizes = bus_sizes_norm, bus_scale = 1e-7, title = 'Gas energy balance')

In [ ]:
n.links_t.p['Industry_boiler'].iloc[0], n.links_t.p1['Industry_boiler'].iloc[0], n.links_t.p2['Industry_boiler'].iloc[0], n.links_t.p['Industry_boiler'].max()

In [ ]:
n.links.loc['Industry_boiler']['efficiency'] * n.links.loc['Industry_boiler']['p_nom'] 

In [ ]:
(n.loads_t.p_set['industry_thermal'].round(16) + n.links_t.p1['Industry_boiler'].round(16)).sum()

In [ ]:
n.loads_t.p_set['kritis_thermal'] + n.links_t.p1['Kritis_CHP']

In [ ]:
n.loads_t.p_set['kritis'] + n.links_t.p2['Kritis_CHP']
# electricity demand beign matched, neglecting required thermal

### Component loading

In [ ]:
import numpy as np
from adjustText import adjust_text #to print text without overlaps automatically
import matplotlib.cm as cm
import matplotlib.colors as colors

In [ ]:
def get_carrier_Xlinks(n, carrier):
    """
    Function to get the X-coupling links connected with buses of given carrier.
    considers if either of the two output buses is connected to a bus of the given carrier
    """
    buses = n.buses[n.buses.carrier == carrier].index
    # Transport links have carriers like AC or gas
    Xlinks = n.links.loc[((n.links.carrier != 'AC') & (n.links.carrier != 'gas')) & ((n.links.bus1.isin(buses)) | (n.links.bus2.isin(buses)))]
    return Xlinks

**Setup instruction**
The carrier of transport links have to be set similar to that of the buses it connections.  
Leaving it empty, would make it default to this.  
Xlinks should have carreirs defined by their techs.  


In [ ]:
def plot_network(ax, n, carrier, col_scheme = {'gens' : {}, 'Xlinks' : {}, 'bus' : 'orange', 'links' : {}}, title = None):
    """
    Plots the network of the specified carrier with the specified colour scheme, could also provide colourmaps for the links and generators.
    """
    cr_buses = n.buses[n.buses.carrier == carrier].index
    # Transport lines
    transport_links = n.links.loc[n.links.carrier == carrier]
    #Gen per buses
    gens_per_bus = n.generators.groupby('bus').groups
    cr_gens_per_bus = {bus: gens_per_bus[bus] for bus in cr_buses if bus in gens_per_bus}
    #Xlinks
    Xlinks = n.links.loc[((n.links.carrier != 'AC') & (n.links.carrier != 'gas'))]
    out_buses = Xlinks[['bus1', 'bus2']].stack().reset_index(level = 1, drop = True).rename('out_buses')
    out_buses = out_buses[out_buses != ''] #removing the empty buses
    Xlinks_per_bus = out_buses.groupby(out_buses).groups
    cr_Xlinks_per_bus = {bus: Xlinks_per_bus[bus] for bus in cr_buses if bus in Xlinks_per_bus}

    #Plotting buses
    ax.scatter(n.buses.loc[cr_buses].x, n.buses.loc[cr_buses].y, s = 100, c = col_scheme.get('bus'), label = 'Buses', zorder = 3)

    #Plotting transport links or lines
    for link in transport_links.index:
        ax.plot([transport_links.bus0.map(n.buses.x)[link], transport_links.bus1.map(n.buses.x)[link]], [transport_links.bus0.map(n.buses.y)[link], transport_links.bus1.map(n.buses.y)[link]], c = col_scheme.get('links').get(link, 'grey'), alpha = 0.8)
        # ax.plot([n.buses[n.links[link].bus0].x, n.buses[n.links[link].bus1].x], [n.buses[n.links[link].bus0].y, n.buses[n.links[link].bus1].y], c = col_scheme.get('links').get(link, 'grey'))
    
    texts = [] #all text collexted here, and managed by adjust_text to avoid overlaps
    # offset = 80e-6
    offset = 0
    angles_gens = {}
    angles_xlinks = {}
    for bus in cr_buses:
        gens = gens_per_bus.get(bus)
        Xlinks = Xlinks_per_bus.get(bus)
        
        bus_x = n.buses.loc[bus].x
        bus_y = n.buses.loc[bus].y

        #Annotation for bus
        texts.append(ax.text(bus_x + offset, bus_y + offset, bus, fontsize = 8))

        ## What happens if there are no gens or Xlinks connected to the bus? 
        if gens is not None and len(gens) > 0:
            angles_gens = np.linspace(0, np.pi, len(gens), endpoint=False)
            for gen, angle in zip(gens, angles_gens):
                gen_x = bus_x + 0.0002 * np.cos(angle)
                gen_y = bus_y + 0.0002 * np.sin(angle)
                ax.scatter(gen_x, gen_y, s = 50, c = col_scheme.get('gens').get(gen, 'green'), label = 'Generators')
                ax.plot([bus_x, gen_x], [bus_y, gen_y], c = 'grey', alpha = 0.5)
                texts.append(ax.text(gen_x + offset, gen_y + offset, gen, fontsize = 8))
        
        if Xlinks is not None and len(Xlinks) > 0:
            angles_xlinks = np.linspace(np.pi, 2*np.pi, len(Xlinks), endpoint=False)
            for Xlink, angle in zip(Xlinks, angles_xlinks):
                Xlink_x = bus_x + 0.0002 * np.cos(angle)
                Xlink_y = bus_y + 0.0002 * np.sin(angle)
            ax.scatter(Xlink_x, Xlink_y, s = 50, c = col_scheme.get('Xlinks').get(Xlink, 'teal'), label = 'X-coupling links')
            ax.plot([bus_x, Xlink_x], [bus_y, Xlink_y], c = 'grey', alpha = 0.5)
            texts.append(ax.text(Xlink_x + offset, Xlink_y + offset, Xlink, fontsize = 8))

    adjust_text(texts, ax = ax, arrowprops=dict(arrowstyle='->', color='red'))

    ax.set_xticks([])
    ax.set_yticks([])

    if title is not None:
        ax.set_title(title)

    



In [ ]:
def plot_90th_percentile_loading(n, carrier, ax, title = None):
    """
    Funciton to plot the 90th percentile loading of all components of specified carrier.
    """

    cr_buses = n.buses[n.buses.carrier == carrier].index
    # Transport lines/links loading
    transport_links = n.links.loc[n.links.carrier == carrier]
    transport_links_90_loading = (n.links_t.p1[transport_links.index].abs() / transport_links['p_nom']).quantile(0.9)

    #Sector-coupling links loading
    cr_xlinks = get_carrier_Xlinks(n, carrier)
    cr_xlinks_90_loading = (n.links_t.p1[cr_xlinks.index].abs() / cr_xlinks['p_nom']).quantile(0.9)

    #generator loading
    cr_gens = n.generators.loc[n.generators['bus'].isin(cr_buses)]
    cr_gens_90_loading = (n.generators_t.p[cr_gens.index].abs() / cr_gens['p_nom']).quantile(0.9)

    #Prepping colour map
    cmap = plt.get_cmap('viridis')
    norm = colors.Normalize(vmin=0, vmax=1)

    col_scheme = {}
    col_scheme['gens'] = {gen: cmap(norm(cr_gens_90_loading[gen])) for gen in cr_gens_90_loading.index}
    col_scheme['Xlinks'] = {link: cmap(norm(cr_xlinks_90_loading[link])) for link in cr_xlinks_90_loading.index}
    col_scheme['bus'] = 'orange'  # Keeping bus color constant
    col_scheme['links'] = {link: cmap(norm(transport_links_90_loading[link])) for link in transport_links_90_loading.index}
    
    plot_network(ax, n, carrier, col_scheme = col_scheme, title = title)

    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])   # required by matplotlib

    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label("90th percentile loading")


In [ ]:
fig, ax = plt.subplots(figsize = (15,8))
plot_90th_percentile_loading(n, 'AC', ax, title = 'Electrical network 90th percentile loading')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (15,8))
plot_90th_percentile_loading(n, 'thermal', ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (15,8))
plot_90th_percentile_loading(n, 'gas', ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (15,8))

plot_network(ax, n, 'AC')

In [ ]:
def get_carrieronly_network(n, carrier):
    '''
    Function that would return entities associated with buses of a given carrier.
    Currently the returns are:
    - Buses: all buses of the given carrier
    - gens: all gens connected to buses of the given carrier
    - loads: all loads connected to buses of the given carrier
    - links: all transport links of the given carrier
    - Xlinks: all sector-coupling links connected to buses of the given carrier
    - Stores: all stores connected to buses of the given carrier
    '''

    cr_buses = n.buses[n.buses.carrier == carrier].index
    trans_links = n.links[n.links.carrier == carrier]
    gens = n.generators[n.generators.bus.isin(cr_buses)]
    loads = n.loads[n.loads.bus.isin(cr_buses)]
    Xlinks = get_carrier_Xlinks(n, carrier)
    stores = n.stores[n.stores.bus.isin(cr_buses)]
    slacks = n.generators[(n.generators.carrier == 'Load_shed') & (n.generators.bus.isin(cr_buses))]

    network_slice = {
        'buses': cr_buses,
        'gens': gens,
        'trans_links': trans_links,
        'loads': loads,
        'xlinks': Xlinks,
        'stores': stores,
        'slacks': slacks
    }

    return network_slice

In [ ]:
def compute_comp_loading(n, carrier):
    '''
    Function to compute the loading of all components of a given carrier, and returns a dataframe.
    '''

    cr_network = get_carrieronly_network(n, carrier)
    loading = {}
    loading['gens'] = (n.generators_t.p[cr_network['gens'].index].abs() / cr_network['gens']['p_nom'])
    loading['trans_links'] = (n.links_t.p1[cr_network['trans_links'].index].abs() / cr_network['trans_links']['p_nom'])
    loading['xlinks'] = (n.links_t.p1[cr_network['xlinks'].index].abs() / cr_network['xlinks']['p_nom'])
    
    return loading

In [ ]:
ex = get_carrieronly_network(n, 'AC')
ex['trans_links']['p_nom']

In [ ]:
def prep_colour_sheme(quantity, thresholds = {'Default' : 1}):
    '''
    Function to prepare a colour scheme according to the provided loading and specified thresholds.
    Threshold being used to consider appropriate loading for heat pumps; 
    could also be used to set 0.75 or something as the threshold for transport lines (N-1 criteria)
    '''

    cmap = plt.get_cmap('viridis')
    norm = colors.Normalize(vmin=0, vmax=thresholds.get('Default', 1))

    col_scheme = {
        comp: {
            entity: cmap(norm(quantity[comp][entity])) for entity in quantity[comp].keys()
        }for comp in quantity.keys()
    }
    # now the seperate threshold for heat pump

In [ ]:
#computing component loading

thresholds = {
    'HP' : 2.5
}
carrier = 'thermal'

cr_network = get_carrieronly_network(n, carrier)

#filtering snapshots where load shedding is happening, anywhere

slack_gens = n.generators_t.p[cr_network['slacks'].index]
mask = (slack_gens > 0).any(axis=1)
load_shedding_snaps = n.snapshots[mask]

loading = compute_comp_loading(n, carrier) 

#preparing colour scheme based on thresholds
## 90th percentile loading during load shedding snaps

# shedding_loading_90th = {comp: {entity: loading[comp][entity].loc[load_shedding_snaps].quantile(0.9)} for comp in loading.keys() for entity in loading[comp].keys()}
shedding_loading_90th = {
    comp: {
        entity: loading[comp][entity].loc[load_shedding_snaps].quantile(0.9)
        for entity in loading[comp].columns}
    for comp in loading.keys()}
# or if a particular index
index = '2026-01-03 06:00:00'
idx_loading = {
    comp: {
        entity: loading[comp][entity].loc[index]
        for entity in loading[comp].columns}
    for comp in loading.keys()
    }






In [ ]:
shedding_loading_90th

In [ ]:
loading.keys()

In [ ]:
for comp in loading.keys():
    print(f"Component: {comp}")
    for entity in loading[comp].columns:
        print(f"  Entity: {entity}, Loading: {loading[comp][entity].loc[load_shedding_snaps].quantile(0.9)}")

In [ ]:
shedding_loading_90th

In [ ]:
loading['gens']

In [ ]:
shedding_loading_90th

In [ ]:
load_shedding_snaps

In [ ]:
shedding_loading_90th

In [ ]:
n.generators_t.p

In [ ]:
cr_network['slacks']

# EOF
rough

In [ ]:
n.statistics.energy_balance(bus_carrier = 'AC', groupby = ['bus', 'carrier'])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

elbusdf = n.buses.loc[el_buses]
ax.scatter(elbusdf.x, elbusdf.y)

for bus in elbusdf.index:
    ax.text(
        elbusdf.loc[bus, "x"],
        elbusdf.loc[bus, "y"],
        bus,
        fontsize=8
    )

ax.set_aspect("equal")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

for link in n.links.itertuples():
    x0, y0 = n.buses.loc[link.bus0, ["x", "y"]]
    x1, y1 = n.buses.loc[link.bus1, ["x", "y"]]

    ax.plot(
        [x0, x1],
        [y0, y1],
        color="grey",
        linewidth=1
    )

for bus in n.buses.index:
    ax.text(
        n.buses.loc[bus, "x"],
        n.buses.loc[bus, "y"],
        bus,
        fontsize=8,
        # xytext=(5,5),
        # textcoords="offset points"
    )

for bus in n.buses.index:
    ax.text(
        n.buses.loc[bus, "x"],
        n.buses.loc[bus, "y"],
        bus,
        fontsize=8,
        # xytext=(5,5),
        # textcoords="offset points"
    )

In [ ]:
fig, ax = plt.subplots(
    figsize=(12, 8),
    subplot_kw={"projection": ccrs.Mercator()}
)
n.statistics.energy_balance.plot.map( ax = ax, bus_split_circle = False)

In [ ]:
n.statistics.energy_balance(
    groupby = ["bus", "carrier"],
    components = ['Generator', 'Load', 'Link']
).groupby(['bus', 'carrier']).sum()
# n.statistics.energy_balance()

In [ ]:

import matplotlib.pyplot as plt


fig, ax = plt.subplots(figsize=(8, 8))

n.plot(
    ax=ax,
    geomap=False,          # <-- key fix: don't require cartopy GeoAxes
    bus_sizes=0.0000000000002,
    bus_colors="steelblue",
    line_widths=0,
    link_widths=2,
    link_colors="firebrick",
    title="Network topology",
)

# add bus name annotations
for name, row in n.buses.iterrows():
    ax.annotate(
        name,
        xy=(row.x, row.y),
        xytext=(5, 5),                # offset in points, so it doesn't sit on top of the marker
        textcoords="offset points",
        fontsize=8,
        ha="left",
    )

plt.tight_layout()
plt.show()

In [ ]:
n.stats.energy_balance().plot()